In [ ]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]




#imports

import os
from dotenv import load_dotenv

from IPython.display import Markdown, display
from openai import OpenAI


# Load environment variables in a file called .env


openai= OpenAI()
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')


# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")



# Step 2: Make the messages list

system_prompt = "you are a helpful assistant in science that list 10 relevant sources of scientific publications related to a first article topic. you do this in harvard style. You also summarise the topic given with simple words without gargon. Give a funny and cheeky analogy. "
user_prompt = """
   here is a scientific article. provide some relevant sources. I also want a simple summary of the sources. 
"""

lm = fetch_website_contents("https://pubmed.ncbi.nlm.nih.gov/32325762/")


def messages_for(article):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + article}
    ]# fill this in

messages_for(lm)



# Step 3: Call OpenAI

def provideref(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content
provideref("https://pubmed.ncbi.nlm.nih.gov/32325762/")



# Step 4: print the result

def display_summaryref(url):
    summary = provideref(url)
    display(Markdown(summary))
display_summaryref("https://pubmed.ncbi.nlm.nih.gov/32325762/")    